# Week 8 Overview

This week will be a mix of data joining/merging problems and linear algebra. 
The first 5 problems are data cleaning and the final 4 problems are linear algebra. 

There are multiple ways to combine data. These methods are common cross multiple languages like pandas, SQL, and R. At times the naming is different but the general concepts apply. 

### **Joining or Merging**
This is a process of combining two datasets by adding the columns of one dataset to the other by some logical relationship between the columns. 

In SQL we call this joining but pandas has two functions:

**merge** - The default behavor for merge is to combine on columns matching.

**join** - The default behavor for join is to combine on the column index matching. 

Often times I will colloquially use the word "join" for either merging or joining in pandas. 

Left Dataset
| key    | value |
| -------- | ------- |
| A1  | $250    |
| A2 | $80     |
| A3    | $420    |

Right Dataset
| key    | different_value |
| -------- | ------- |
| A1  | cat    |
| A2 | dog     |
| A3    | apple    |

Data Joined on key

| key    | value | different_value |
| -------- | ------- | ------- |
| A1  | $250    | cat |
| A2 | $80     | dog |
| A3    | $420    | apple |

Typically we refer to the starting dataset as the left dataset and the one being added as the right. 

The logic is typically that there is the same value in a specific column in both datasets. SQL allows for slightly more advanced logic which we will learn next quarter. Today we will focus on just columns matching. 

There are different types of joins that you will explore in this notebook (inner, outer, left, right, cross). The typical visual that is used to illustration these concepts in Ven Diagrams. If you are getting stuck trying to pick the right join type search for "types of joins" and look at the pictures that come up.


## **Concat or Union**

This is a process of combining two dataset by adding the rows of one dataset to the end of another. There is no logic required for this. This is called conact in pandas and union in SQL. 

In most version of SQL you are required to have the same columns in both datasets. In pandas you don't have to. If I concatenate the two dataset above in pandas I would get:

| key    | value | different_value |
| -------- | ------- | ------- |
| A1  | $250    | null |
| A2 | $80     | null |
| A3    | $420    | null |
| A1  | null    | cat |
| A2 | null     | dog |
| A3    | null    | apple |

However if my right dataset looked like this:

| key    | value |
| -------- | ------- |
| A1  | cat    |
| A2 | dog     |
| A3    | apple    |

then I could union them in SQL or concat in pandas to get:

| key    | value |
| -------- | ------- |
| A1  | $250    |
| A2 | $80     |
| A3    | $420    |
| A1  | cat    |
| A2 | dog     |
| A3    | apple    |



In [1]:
import pandas as pd
import numpy as np

In [2]:
df_1 = pd.DataFrame({"ints": range(100)})
df_2 = pd.DataFrame({"ints": range(-10, 10)}, index=range(-10, 10))


df_1['threes'] = np.floor(df_1['ints']/3) * 3

df_2['evens'] = df_2['ints']*2
df_2['threes'] = np.floor(df_2['ints']/3) * 3

### Problem 1:

Your first task will be to create a dataset by `merging` `df_1` and `df_2` on the `ints` column where the match on both sides. The results will be a dataframe with 10 rows and 5 columns.

You will then create the same dataframe by using the `join` function and joining the two datasets where the indexes are equal. There will be a little more work of handle column duplication so look up the error and figure out arguments to set. How many columns do you get in this case?

In [3]:
merged_df = df_1.merge(df_2, on="ints")
merged_df

,ints,threes_x,evens,threes_y
0,0,0.0,0,0.0
1,1,0.0,2,0.0
2,2,0.0,4,0.0
3,3,3.0,6,3.0
4,4,3.0,8,3.0
5,5,3.0,10,3.0
6,6,6.0,12,6.0
7,7,6.0,14,6.0
8,8,6.0,16,6.0
9,9,9.0,18,9.0


In [12]:
df1_indexed = df_1.set_index("ints")
df2_indexed = df_2.set_index("ints")

joined_df = df1_indexed.join(df2_indexed, how="inner", lsuffix="_left",
    rsuffix="_right")

## Problem 2:

Next you will perform the same merge as above three times with the following modifications:

* You want to keep all rows in `df_1` even if there is no match found in `df_2`
* You want to keep all rows in `df_2` even if there is no match found in `df_1`
* You want to keep all rows in `df_1` and `df_2` even if there is no match found in the other dataframe


How many rows do you end up with in each case? 

Think through a scenario where you might want to do this and add it as a comment above each merge. 

In [5]:
# Scenario: Keep every record from df_1 
left_merge = df_1.merge(df_2, on="ints", how="left")
left_rows = left_merge.shape[0]

left_rows

100

In [6]:
# Scenario: Keep every record from df_2 
right_merge = df_1.merge(df_2, on="ints", how="right")
right_rows = right_merge.shape[0]



In [7]:
# Scenario: Combining two systems and want to keep everything from both sources.
outer_merge = df_1.merge(df_2, on="ints", how="outer")
outer_rows = outer_merge.shape[0]

left_rows, right_rows, outer_rows

(100, 20, 110)

### Problem 3

Now we are going to merge on columns that are not the same. Merge on the following:

* Merge `df_1` and `df_2` where `df_1.ints = df_2.evens`, only keep rows where there is a value for either dataframe
* Merge `df_1` and `df_2` where `df_1.ints = df_2.threes`, only keep rows where there is a value for either dataframe
* Merge `df_1` and `df_2` where `df_1.ints = df_2.threes`, keep all rows from `df_1` even if there is no match found in `df_2`
* Merge `df_1` and `df_2` where `df_1.threes = df_2.threes`, only keep rows where there is a value for either dataframe


How many rows do you end up with in each case? Are there any duplications? (try: value_count)

Think through a scenario where you might want to do this and add it as a comment above each merge. 

In [17]:
m1 = df_1.merge(
    df_2,
    left_on="ints",
    right_on="evens",
    how="outer",
    indicator=True 
)

print("Merge 1 rows:", m1.shape[0])
print("m1 columns:", m1.columns.tolist())
# Duplication check: do we see repeated keys?
print("Dupes in df_1.ints within merged (ints_x):", m1["ints_x"].value_counts().head())
print("Dupes in df_2.evens within merged:", m1["evens"].value_counts().head())

Merge 1 rows: 110
m1 columns: ['ints_x', 'threes_x', 'ints_y', 'evens', 'threes_y', '_merge']
Dupes in df_1.ints within merged (ints_x): ints_x
0.0    1
1.0    1
2.0    1
3.0    1
4.0    1
Name: count, dtype: int64
Dupes in df_2.evens within merged: evens
-20.0    1
-18.0    1
-16.0    1
-14.0    1
-12.0    1
Name: count, dtype: int64


In [16]:
m2 = df_1.merge(
    df_2,
    left_on="ints",
    right_on="threes",
    how="outer",
    indicator=True
)

print("Merge 2 rows:", m2.shape[0])
print(m2["_merge"].value_counts())

# In this merge, keys from df_1 and df_2 are in different columns, may be suffixed
print("m2 columns:", m2.columns.tolist())
print("Dupes in df_1.ints within merged (ints_x):", m2["ints_x"].value_counts().head())
print("Dupes in df_2.threes within merged (threes_y):", m2["threes_y"].value_counts().head())

Merge 2 rows: 116
_merge
left_only     96
right_only    10
both          10
Name: count, dtype: int64
m2 columns: ['ints_x', 'threes_x', 'ints_y', 'evens', 'threes_y', '_merge']
Dupes in df_1.ints within merged (ints_x): ints_x
0.0    3
3.0    3
6.0    3
1.0    1
2.0    1
Name: count, dtype: int64
Dupes in df_2.threes within merged (threes_y): threes_y
-9.0    3
-6.0    3
-3.0    3
 0.0    3
 6.0    3
Name: count, dtype: int64


In [ ]:

m3 = df_1.merge(
    df_2,
    left_on="ints",
    right_on="threes",
    how="left",
    indicator=True
)
print(m3.columns.tolist())

print("Merge 3 rows:", m3.shape[0])
print(m3["_merge"].value_counts())



['ints_x', 'threes_x', 'ints_y', 'evens', 'threes_y', '_merge']
Merge 3 rows: 106
_merge
left_only     96
both          10
right_only     0
Name: count, dtype: int64


In [ ]:
# Scenario: both datasets share a common grouping key (threes),
# but each has different attributes. You want the union of groups across both sources.
m4 = df_1.merge(
    df_2,
    on="threes",
    how="outer",
    indicator=True
)

print("Merge 4 rows:", m4.shape[0])
print(m4["_merge"].value_counts())



Merge 4 rows: 128
_merge
left_only     88
both          30
right_only    10
Name: count, dtype: int64


### Problem 4

Add a new the column to `df_2` called `threes_string` that is the `threes` column converted to a string. Attempt to merge `df_1` and `df_2` where `df_1.threes = df_2.threes_string` with an inner join. What happens? Why?

In [ ]:
df_2["threes_string"] = df_2["threes"].astype(str)
mismatch_merge = df_1.merge(
    df_2,
    left_on="threes",
    right_on="threes_string",
    how="inner"
)

#error occurs because the data types are not exact matches.

ValueError: You are trying to merge on float64 and object columns for key 'threes'. If you wish to proceed you should use pd.concat

### Problem 5

Now you will play around with `pd.concat` by doing the following:

* Concatenate `df_1` and `df_2` keeping all rows, columns and indexes
* Concatenate `df_1` and `df_2` keeping all rows and columns but ignore the indexes from the orginal dataframes and instead have the index on this dataframe be zero to the number of rows.
* Concatenate `df_1` and `df_2` keeping all rows and indexes the same but only keeping columns that exist in both dataframes


In [ ]:
# 1) Keep all rows, columns, and indexes (default behavior)
df_all = pd.concat([df_1, df_2])
df_all

# 2) Keep all rows and columns, but reset index to 0..n-1
df_new_index = pd.concat([df_1, df_2], ignore_index=True)
df_new_index

# 3) Keep all rows and original indexes, but ONLY keep columns that exist in both dataframes
df_common_cols = pd.concat([df_1, df_2], join="inner")
df_common_cols

,ints,threes
0,0,0.0
1,1,0.0
2,2,0.0
3,3,3.0
4,4,3.0
...,...,...
5,5,3.0
6,6,6.0
7,7,6.0
8,8,6.0


## Linear Algebra: Rank and Column Space

### Problem 6
You will now learn how to create random matrices with arbitrary rank (subject to the constraints about matrix sizes, etc.). To create an $m \times n$ matrix with rank $r$, multiply a random $m \times r$ matrix with a random $r \times n$ matrix. Implement this in Python and confirm that the rank is indeed $r$. 

What happens if you set $r > min{M,N}$, and why does that happen?

In [14]:
import numpy as np

m, n, r = 6, 5, 3

A = np.random.rand(m, r)   # m x r
B = np.random.rand(r, n)   # r x n

M = A @ B                  # m x n matrix

print("Rank:", np.linalg.matrix_rank(M))

Rank: 3


### Problem 7
Interestingly, the matrices $A$, $A^T$, $A^T A$, and $AA^T$ all have the same rank. Write code to demonstrate this, using random matrices of various sizes, shapes (square, tall, wide), and ranks. Create a total of 6 random, two of each size that have different sizes and ranks. 

In [18]:

def matrix_ranks(A):
    rA = np.linalg.matrix_rank(A)
    rAT = np.linalg.matrix_rank(A.T)
    rATA = np.linalg.matrix_rank(A.T @ A)
    rAAT = np.linalg.matrix_rank(A @ A.T)
    return rA, rAT, rATA, rAAT

cases = [
    (5, 5, 4),
    (5, 5, 2),
    (6, 4, 3),
    (6, 4, 2),
    (4, 6, 3),
    (4, 6, 2)
]

for m, n, r in cases:
    U = np.random.randn(m, r)
    V = np.random.randn(r, n)
    A = U @ V
    rA, rAT, rATA, rAAT = matrix_ranks(A)
    print(f"A: {m}x{n}, target rank {r}, actual rank {rA}")
    print(f"A.T rank {rAT}, A.TA rank {rATA}, AA.T rank {rAAT}")
    assert rA == r, "actual rank may differ if random factors approximately degenerate"
    assert rA == rAT == rATA == rAAT
    print("---")

A: 5x5, target rank 4, actual rank 4
A.T rank 4, A.TA rank 4, AA.T rank 4
---
A: 5x5, target rank 2, actual rank 2
A.T rank 2, A.TA rank 2, AA.T rank 2
---
A: 6x4, target rank 3, actual rank 3
A.T rank 3, A.TA rank 3, AA.T rank 3
---
A: 6x4, target rank 2, actual rank 2
A.T rank 2, A.TA rank 2, AA.T rank 2
---
A: 4x6, target rank 3, actual rank 3
A.T rank 3, A.TA rank 3, AA.T rank 3
---
A: 4x6, target rank 2, actual rank 2
A.T rank 2, A.TA rank 2, AA.T rank 2
---


### Problem 8

Demonstrate the addition rule of matrix rank $(r(A + B) ≤ r(A) + r(B))$ by creating three pairs of rank-1 matrices that have a sum with 
1. rank-0
2. rank-1
3. rank-2

Then repeat this exercise using matrix multiplication instead of addition.

In [23]:

# helper for rank-1 matrix
def rank1(u, v):
    return np.outer(u, v)

u = np.random.randn(4)
v = np.random.randn(4)
A = rank1(u, v)

# 1) A + (-A) => rank 0
B = -A
print("case 1 (sum rank0):", "rank(A)=", np.linalg.matrix_rank(A), "rank(B)=", np.linalg.matrix_rank(B), "rank(A+B)=", np.linalg.matrix_rank(A+B))

# 2) A + 2A => rank 1
B = 2 * A
print("case 2 (sum rank1):", "rank(A)=", np.linalg.matrix_rank(A), "rank(B)=", np.linalg.matrix_rank(B), "rank(A+B)=", np.linalg.matrix_rank(A+B))

# 3) A + B where B is independent rank-1 => rank 2
u2 = np.random.randn(4)
v2 = np.random.randn(4)
B = rank1(u2, v2)
print("case 3 (sum rank2):", "rank(A)=", np.linalg.matrix_rank(A), "rank(B)=", np.linalg.matrix_rank(B), "rank(A+B)=", np.linalg.matrix_rank(A+B))

# multiplication for rank-1 matrices
print("prod case1: rank(A @ -A)=", np.linalg.matrix_rank(A @ (-A)))
print("prod case2: rank(A @ (2A))=", np.linalg.matrix_rank(A @ (2*A)))
print("prod case3: rank(A @ B)=", np.linalg.matrix_rank(A @ B))

case 1 (sum rank0): rank(A)= 1 rank(B)= 1 rank(A+B)= 0
case 2 (sum rank1): rank(A)= 1 rank(B)= 1 rank(A+B)= 1
case 3 (sum rank2): rank(A)= 1 rank(B)= 1 rank(A+B)= 2
prod case1: rank(A @ -A)= 1
prod case2: rank(A @ (2A))= 1
prod case3: rank(A @ B)= 1


### Problem 9

The goal of this exercise is to answer the question is $v \in C(A)$?

Create a rank-3 matrix $A \in \mathbb{R}^{4 \times 3}$ and vector $v \in \mathbb{R}^{4}$ using numbers randomly drawn from a normal distribution. 

Follow the algorithm described in the [In the Column Space?](https://learning.oreilly.com/library/view/practical-linear-algebra/9781098120603/ch06.html#id335) section of Practical Linear Algebra to determine whether the vector is in the column space of the matrix. 

Rerun the code multiple times to see whether you find a consistent pattern. 

Next, use a $A \in \mathbb{R}^{4 \times 4}$ rank-4 matrix and a vector $v \in \mathbb{R}^{4}$ using numbers randomly drawn from a normal distribution. What happens in this case? Why?


In [27]:
# Problem 9: Check v in column space of A
# Part 1: non-square 4x3 rank-3
A = np.random.randn(4, 3)
while np.linalg.matrix_rank(A) != 3:
    A = np.random.randn(4, 3)

v = np.random.randn(4)

x, residuals, _, _ = np.linalg.lstsq(A, v, rcond=None)
res = np.linalg.norm(A @ x - v)
print("A shape", A.shape, "rank", np.linalg.matrix_rank(A))
print("residual ", res)
print("column space", res < 1e-8)

# run multiple trials to see pattern
for i in range(5):
    v = np.random.randn(4)
    x, residuals, _, _ = np.linalg.lstsq(A, v, rcond=None)
    res = np.linalg.norm(A @ x - v)
    print(f"trial {i}", "res", res, "in C(A)", res < 1e-8)

# Part 2: square full-rank 4x4
A_full = np.random.randn(4, 4)
while np.linalg.matrix_rank(A_full) != 4:
    A_full = np.random.randn(4, 4)

v = np.random.randn(4)
x = np.linalg.solve(A_full, v)
res = np.linalg.norm(A_full @ x - v)
print("A_full shape", A_full.shape, "rank", np.linalg.matrix_rank(A_full))
print("residual norm", res)
# pattern identified is that the column space is always false 


A shape (4, 3) rank 3
residual  1.0406078561638215
column space False
trial 0 res 0.2948195176684553 in C(A) False
trial 1 res 0.6706671020296477 in C(A) False
trial 2 res 0.6427340435550731 in C(A) False
trial 3 res 0.908334312164585 in C(A) False
trial 4 res 0.9575168514687042 in C(A) False
A_full shape (4, 4) rank 4
residual norm 1.374689277502871e-15
